# Linear Transformations

| Difficulty | ███░░░░░░░ 3/10 |
| :--- | :--- |
| Prerequisites | 01.02 |
| Time | ~65 min |
| Colab GPU | Not needed |

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NiravRVaghasiya/maths-for-ai/blob/main/01_Linear_Algebra/03_linear_transformations.ipynb)

---

## 🎯 Learning Objective

See linear maps geometrically — rotation, scaling, shearing — and learn how changing basis changes the coordinates of a transformation without changing the transformation itself.

---

## 📐 Theory

A function $T$ is a **linear transformation** if it respects addition and scaling:

$$T(a\mathbf{u} + b\mathbf{v}) = aT(\mathbf{u}) + bT(\mathbf{v})$$

Every linear transformation on $\mathbb{R}^n$ can be represented by a matrix, and applying the
transformation is just matrix-vector multiplication: $T(\mathbf{x}) = A\mathbf{x}$.

### The standard building blocks

$$R(\theta) = \begin{pmatrix}\cos\theta & -\sin\theta \\ \sin\theta & \cos\theta\end{pmatrix}
\qquad
S(s_x, s_y) = \begin{pmatrix}s_x & 0 \\ 0 & s_y\end{pmatrix}
\qquad
\text{Shear} = \begin{pmatrix}1 & k \\ 0 & 1\end{pmatrix}$$

Rotation preserves lengths and angles; scaling stretches/compresses along the axes
independently; shear slides one axis proportionally to the other, distorting angles while
preserving area.

### Composition

Applying $T_1$ then $T_2$ is itself linear, and its matrix is the **product** $A_2 A_1$ (note
the order: the transformation applied *first* goes on the *right*). Because matrix
multiplication is not commutative in general, **order matters** — rotating then scaling is
usually not the same as scaling then rotating.

### Change of basis

A vector's coordinates depend on which basis you describe it in. If $B = [\mathbf{b}_1 \mid
\mathbf{b}_2 \mid \cdots]$ collects a new basis's vectors as columns, then a vector with
coordinates $\mathbf{c}$ in that basis has standard coordinates $\mathbf{x} = B\mathbf{c}$, so
converting standard coordinates *into* the new basis is $\mathbf{c} = B^{-1}\mathbf{x}$. The
transformation itself doesn't change — only the numbers describing it do.

---

## 👁️ Visual Intuition

Watching a grid of points get rotated, scaled, and sheared makes each transformation's
geometric signature obvious: rotation spins the grid rigidly, scaling stretches it
axis-by-axis, and shear tilts it like a deck of cards.

In [ ]:
# Rotation, scaling, and shear applied to a grid of points
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid')

def rotation_matrix(theta):
    c, s = np.cos(theta), np.sin(theta)
    return np.array([[c, -s], [s, c]])

def scaling_matrix(sx, sy):
    return np.array([[sx, 0.0], [0.0, sy]])

def shear_matrix(k):
    return np.array([[1.0, k], [0.0, 1.0]])

xs, ys = np.meshgrid(np.linspace(-1, 1, 9), np.linspace(-1, 1, 9))
grid = np.stack([xs.ravel(), ys.ravel()])

transforms = {
    "Rotation (45deg)": rotation_matrix(np.pi / 4),
    "Scaling (2x, 0.5x)": scaling_matrix(2.0, 0.5),
    "Shear (k=0.8)": shear_matrix(0.8),
}

fig, axes = plt.subplots(1, 3, figsize=(13, 4.5))
for ax, (title, A) in zip(axes, transforms.items()):
    original = grid
    transformed = A @ grid
    ax.scatter(*original, s=6, color="lightgray", label="original")
    ax.scatter(*transformed, s=10, color="#4C72B0", label="transformed")
    ax.set_title(title)
    ax.set_aspect('equal')
    ax.set_xlim(-2.5, 2.5); ax.set_ylim(-2.5, 2.5)
    ax.legend(fontsize=7)
plt.tight_layout()
plt.show()

## 🐍 Implementation from Scratch

Composition of transformations is matrix multiplication — but order matters. We verify this
directly by applying two transformations sequentially versus multiplying their matrices in
each possible order.

In [ ]:
# Composition = matrix multiplication, and order changes the result
import numpy as np

def rotation_matrix(theta):
    c, s = np.cos(theta), np.sin(theta)
    return np.array([[c, -s], [s, c]])

def scaling_matrix(sx, sy):
    return np.array([[sx, 0.0], [0.0, sy]])

R = rotation_matrix(np.pi / 3)
Sc = scaling_matrix(2.0, 0.5)
x = np.array([1.0, 0.0])

# "Scale, then rotate" means apply Sc first, then R -- matrices compose as R @ Sc
scale_then_rotate = R @ (Sc @ x)
combined_1 = (R @ Sc) @ x
print("Scale-then-rotate matches (R @ Sc) @ x:", np.allclose(scale_then_rotate, combined_1))

# "Rotate, then scale" is the OPPOSITE order: Sc @ R
rotate_then_scale = Sc @ (R @ x)
combined_2 = (Sc @ R) @ x
print("Rotate-then-scale matches (Sc @ R) @ x:", np.allclose(rotate_then_scale, combined_2))

print("\nAre the two orders the same result?", np.allclose(scale_then_rotate, rotate_then_scale))
print("scale-then-rotate:", np.round(scale_then_rotate, 4))
print("rotate-then-scale:", np.round(rotate_then_scale, 4))
print("Matrix multiplication is NOT commutative in general -- order of operations matters.")

## 🤖 Why This Matters for AI

Every layer of a neural network is (at minimum) a linear transformation followed by a
nonlinearity. Stacking layers stacks these warps of space. Below, we push a synthetic
two-class dataset that is **not** linearly separable through two random (untrained) linear +
ReLU layers, purely to see the geometric *mechanism* at work. With **trained** weights, this
warping becomes dramatic enough to make the classes linearly separable by the final layer —
this exact picture is why deep networks can solve problems a single linear classifier cannot.

In [ ]:
# Watching a linear map + nonlinearity warp a 2D point cloud (untrained weights -- illustrative)
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_moons

X, y = make_moons(n_samples=200, noise=0.08, random_state=0)

rng = np.random.default_rng(1)
W1 = rng.standard_normal((2, 2))
b1 = rng.standard_normal(2) * 0.1
W2 = rng.standard_normal((2, 2))
b2 = rng.standard_normal(2) * 0.1

def relu(z):
    return np.maximum(0.0, z)

H = relu(X @ W1.T + b1)     # layer 1: linear map + ReLU nonlinearity
Z = H @ W2.T + b2            # layer 2: another linear map

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, data, title in zip(axes, [X, H, Z],
                            ["Input space\n(not linearly separable)", "After layer 1\n(linear + ReLU)", "After layer 2"]):
    ax.scatter(data[:, 0], data[:, 1], c=y, cmap="coolwarm", s=15, edgecolor='k', linewidth=0.2)
    ax.set_title(title, fontsize=10)
plt.tight_layout()
plt.show()
print("With RANDOM weights the classes are still tangled -- training adjusts W1, W2, b1, b2")
print("specifically so that the final space becomes linearly separable by a simple boundary.")

## 🏋️ Exercises

### Warm-up
1. Verify computationally that a rotation matrix $R(\theta)$ satisfies $R^\top R = I$ (it's orthogonal) for $\theta = 30°$ and $\theta = 117°$.

### Practice
2. Using `rotation_matrix` and `scaling_matrix`, apply a scale of $(3, 1)$ followed by a $90°$ rotation to the vector $(1, 0)$, two ways: sequential application and a single combined matrix. Confirm they match.

### Challenge
3. Given a new basis $\mathbf{b}_1 = (2,1)$, $\mathbf{b}_2 = (1,1)$, write the matrix $B$ with these as columns. Take the vector $\mathbf{x}=(5,3)$ in standard coordinates, compute its coordinates $\mathbf{c}$ in the new basis via $\mathbf{c}=B^{-1}\mathbf{x}$, then verify $B\mathbf{c}$ recovers $\mathbf{x}$ exactly. Explain why this round-trip is exactly what happens when attention projects tokens into and back out of per-head subspaces.

---

## 📚 Further Reading
- Strang, *Introduction to Linear Algebra*, Ch. 2 (Solving Linear Equations) & Ch. 6 (change of basis)
- [3Blue1Brown: Linear transformations and matrices](https://www.3blue1brown.com/lessons/linear-transformations)

---

*Next notebook: [Systems of Linear Equations](04_systems_of_equations.ipynb)*